# Agronomic Vision & Advisory Engine — end-to-end pipeline

Leaf photograph + local field conditions -> **disease classification**, a **quantitative
severity index**, **retrieval** over agricultural extension material, and a
**severity-grounded treatment plan**.

The architectural claim this notebook demonstrates: the advisory LLM is never allowed to
free-associate, because it receives three independent hard signals.

| # | Signal | Source | Answers |
|---|--------|--------|---------|
| 1 | Disease class + confidence | PyTorch ResNet-50 | *What* is it? |
| 2 | **Severity index (% leaf area)** | OpenCV colour masking — no neural network | *How bad* is it? |
| 3 | Local context | Farmer input via the PWA | *Where and when*? |

Signal 2 is the load-bearing one. It maps to an **action tier**, and that tier is injected
into the prompt as a constraint the model must obey. A 3 % lesion ratio forces
"monitor and sanitise"; a 40 % ratio forces "immediate intervention". This is what stops a
fungicide recommendation for a nearly healthy leaf.

> **Runs on a bare clone.** No trained checkpoint, no Postgres and no API key are required.
> Every missing dependency degrades and is reported in the readiness matrix rather than
> raising. If you have not run the bootstrap yet, run `scripts/bootstrap.ps1` first.

In [1]:
from _bootstrap import setup, show_image

ROOT = setup()

import matplotlib.pyplot as plt
import numpy as np

from backend.app.config import get_settings
from backend.app.core.logging import configure_logging

configure_logging("WARNING")   # keep the notebook output readable
settings = get_settings()

store = "pgvector" if settings.database_url else "local file store"
print(f"device          : {settings.resolve_device()}")
print(f"checkpoint      : {settings.checkpoint_file.name} (exists: {settings.checkpoint_file.exists()})")
print(f"embedding model : {settings.embedding_model}")
print(f"vector store    : {store}")
print(f"llm provider    : {settings.llm_provider}")

project root : d:\MinorProject\Agronomic-Vision-Advisory-Engine
python       : 3.10.5
device          : cuda
checkpoint      : resnet50_plantvillage.pt (exists: True)
embedding model : BAAI/bge-small-en-v1.5
vector store    : local file store
llm provider    : auto


## 0. Build the pipeline

`AgronomicPipeline` is the single orchestrator that the FastAPI routes also use — the
notebook and the API cannot drift apart. Construction loads ResNet-50 and the sentence
transformer once (a few seconds; the first run downloads ~100 MB of ImageNet weights).

In [2]:
from backend.app.services.pipeline import AgronomicPipeline

pipeline = AgronomicPipeline(settings)
readiness = pipeline.readiness()

print("READINESS")
print(f"  checkpoint loaded : {readiness.checkpoint_loaded}")
print(f"  vector store      : {readiness.vector_store} ({readiness.documents_indexed} nodes)")
print(f"  llm provider      : {readiness.llm_provider}")
if readiness.degraded:
    print("\nDEGRADED — the pipeline still runs, with these caveats:")
    for note in readiness.degraded:
        print(f"  - {note}")
else:
    print("\nAll components fully loaded.")

18:29:13 WARNING huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\MinorProject\Agronomic-Vision-Advisory-Engine\.venv\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\AppData\Local\llama_index\llama_index\Cache\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

18:29:33 WARNING app.rag.index | No index found. Run scripts/ingest_knowledge.py to populate it.
READINESS
  checkpoint loaded : True
  vector store      : empty (0 nodes)
  llm provider      : template

DEGRADED — the pipeline still runs, with these caveats:
  - knowledge base empty — advisory is ungrounded
  - no LLM key — deterministic template advisor in use


### Sample input

`scripts/make_samples.py` renders synthetic leaves whose damaged fraction is known *by
construction* — one per action tier. That is what makes the severity engine testable
before the 1.6 GB PlantVillage download, and it is how `tests/test_severity.py` validates
the measurement rather than eyeballing it.

Swap in your own photo by pointing `IMAGE_PATH` at any JPEG or PNG.

In [ ]:
from backend.app.vision.preprocess import read_image

SAMPLES = sorted((ROOT / "data" / "samples").glob("*.png"))
if not SAMPLES:
    raise SystemExit("No samples found. Run: python scripts/make_samples.py")

IMAGE_PATH = ROOT / "data" / "samples" / "leaf_moderate.png"
image = read_image(str(IMAGE_PATH))

print(f"available samples: {[p.stem for p in SAMPLES]}")
print(f"using            : {IMAGE_PATH.name}  {image.shape[1]}x{image.shape[0]}")
show_image(image, IMAGE_PATH.stem, size=(5, 4));

## 1. Severity index — the quantitative signal

Deliberately **not** a neural network. Every number below is a pixel count under a stated
HSV threshold, so it is reproducible and an agronomist can audit it.

The order of operations matters and is not the obvious one:

1. Resize and denoise (bilateral — kills JPEG speckle without smearing lesion edges).
2. **Segment the leaf** on the *raw* image: a saturation/value gate, morphological
   open/close, largest connected component, fill holes. The gate threshold is *adaptive* —
   two candidates (a fixed floor and an Otsu threshold) are validated on plausible area and
   a border test. A fixed floor alone lets a warm-tinted or soil background into the mask,
   which costs about +49 pp of severity; `01_severity_lab.ipynb` measures exactly that.
3. **Then** white balance, using the *background* the segmentation just identified.
4. Classify each pixel inside the leaf: healthy green / chlorotic (yellowing) /
   necrotic (brown-dead).
5. Area ratios, lesion count, largest lesion, action tier.

Why balance *after* segmenting: both textbook illuminant estimators fail on this image.
Grey-world equalises channel means, so a green-dominated frame drags the means and tints
the neutral background up to saturation ~58 — above the leaf threshold — and the mask
swallows the whole frame. White-patch takes gains from a high percentile, so bright yellow
chlorotic tissue outranks the background and the same thing happens. Both break because
the subject is the strongest colour present. Estimating the illuminant from the non-subject
pixels sidesteps it entirely.

In [ ]:
from backend.app.vision.severity import analyse, render_overlay

report, masks = analyse(image)

print(report.to_prompt_block())
print(f"\nleaf coverage of frame : {report.leaf_coverage_pct:.1f} %")
print(f"largest single lesion  : {report.largest_lesion_pct:.1f} % of the leaf")
for warning in report.quality_warnings:
    print(f"  [quality] {warning}")

print(f"\n=> ACTION TIER: {report.action_tier.value.upper()}  ({report.action_tier.headline})")
print(f"   mandate: {report.action_tier.mandate}")

### The figure that makes the number believable

Green = healthy, yellow = chlorotic, red = necrotic, dimmed = outside the leaf. If the
overlay does not match what your eye sees, the severity number is wrong — this panel is
the check, and it is also what `GET /api/v1/severity/overlay` returns to the app so a
farmer can see *why*.

In [ ]:
panels = [
    (masks.working, "1. preprocessed"),
    (masks.leaf, "2. leaf mask"),
    (masks.healthy, "3. healthy green"),
    (masks.necrotic, "4. necrotic"),
    (masks.lesions, "5. lesions (cleaned)"),
    (render_overlay(masks), "6. overlay"),
]

figure, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (panel, title) in zip(axes.ravel(), panels, strict=True):
    show_image(panel, title, ax=ax)
figure.suptitle(
    f"{IMAGE_PATH.stem} — severity {report.severity_pct:.1f} %  "
    f"({report.lesion_count} lesions)  ->  {report.action_tier.value.upper()}",
    fontsize=13,
)
figure.tight_layout()

### Does the measurement actually measure?

The synthetic generator knows the true damaged fraction, so we can plot measured against
truth instead of asserting accuracy. Points should sit on the diagonal.

In [ ]:
from backend.app.vision.synthetic import make_leaf

truth, measured = [], []
for fraction in np.linspace(0.0, 0.7, 12):
    leaf = make_leaf(damaged_fraction=float(fraction),
                     lesion_count=max(int(fraction * 60), 1), seed=13)
    result, _ = analyse(leaf.image)
    truth.append(leaf.true_damaged_fraction * 100)
    measured.append(result.severity_pct)

error = np.abs(np.array(measured) - np.array(truth))
figure, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 75], [0, 75], "--", color="grey", linewidth=1, label="perfect agreement")
for lo, hi, name in [(0, 2, "negligible"), (2, 10, "early"), (10, 25, "moderate"),
                     (25, 50, "severe"), (50, 75, "critical")]:
    ax.axhspan(lo, hi, alpha=0.06, color="red" if hi > 25 else "orange")
    ax.text(1, (lo + hi) / 2, name, fontsize=7, va="center", color="#555")
ax.scatter(truth, measured, s=55, color="#2e7d32", zorder=3, label="synthetic leaves")
ax.set_xlabel("true damaged leaf area (%)")
ax.set_ylabel("measured severity index (%)")
ax.set_title(f"Severity index vs. ground truth\nmean abs error {error.mean():.2f} pp, max {error.max():.2f} pp")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.2);

## 2. Disease classification

`torchvision` ResNet-50, ImageNet-pretrained backbone, `fc` replaced with a 38-class
PlantVillage head.

**If no checkpoint exists**, the head is randomly initialised. The model still runs, but
the label is a placeholder and the pipeline stamps `checkpoint_loaded=False` and
`confidence_reliable=False`. Nothing downstream can mistake noise for a diagnosis: the
prompt marks it PROVISIONAL, and the fallback plan leads with "confirm the identification
before treating". Train it with `02_train_resnet50.ipynb`.

In [ ]:
diagnosis = pipeline.classifier.predict(image, top_k=4)

status = "TRAINED" if diagnosis.checkpoint_loaded else "UNTRAINED HEAD — label is a placeholder"
print(f"[{status}]\n")
print(f"  {diagnosis.top.crop} / {diagnosis.top.disease}   {diagnosis.top.confidence:.1%}")
for alternative in diagnosis.alternatives:
    print(f"    alt: {alternative.crop} / {alternative.disease}  {alternative.confidence:.1%}")
print(f"\n  healthy class     : {diagnosis.is_healthy}")
print(f"  confidence usable : {diagnosis.confidence_reliable} "
      f"(floor {settings.classifier_confidence_floor:.0%})")

## 3. Retrieval — grounding on all three signals

The query is assembled from disease **and** severity tier **and** local context, because a
document about *managing established late blight in wet conditions* should outrank one
about *identifying late blight* when severity is already 40 %.

Retrieval is vector top-k (local `bge-small` embeddings, 384-dim, free and offline) over
pgvector — or the on-disk store when Postgres is not up — followed by a lexical rerank.
The rerank matters on a small corpus, where pure cosine similarity is noisy: a hard
overlap bonus on the actual crop and disease name pulls the right bulletin to the front.

In [ ]:
from backend.app.rag.retriever import build_query, retrieve_evidence
from backend.app.schemas.environment import EnvironmentContext, IrrigationType, Season, SoilType

environment = EnvironmentContext(
    region="Nashik, Maharashtra",
    soil_type=SoilType.BLACK,
    season=Season.KHARIF,
    irrigation=IrrigationType.SPRINKLER,
    rainfall_last_7d_mm=118,
    temperature_c=24,
    relative_humidity_pct=88,
    soil_ph=7.4,
    crop_stage="fruiting",
    days_since_last_spray=21,
    field_area_ha=0.8,
    organic_only=False,
)

print("QUERY SENT TO THE VECTOR STORE")
print(f"  {build_query(diagnosis, report, environment)}\n")

evidence, _ = retrieve_evidence(pipeline.knowledge, diagnosis, report, environment, settings)
for item in evidence:
    snippet = item.text[:200].replace("\n", " ")
    print(f"[{item.id}] score {item.score:.3f}  {item.source}")
    print(f"      {item.crop} / {item.disease} ({item.doc_type})")
    print(f"      {snippet}...\n")

## 4. Synthesis — where severity becomes a constraint

Three things make this more than "ask an LLM about a plant":

1. The measured tier is stated as a **mandate** the model must obey.
2. Retrieved chunks are **numbered evidence**; the model must cite by id.
3. `_strip_uncited_chemicals` **deletes** any chemical recommendation carrying a number
   (a rate, a concentration, an interval) with no citation.

Point 3 is enforced in code, not by asking nicely. Hallucinating a pesticide dose is the
one failure mode in this project that could damage a crop or hurt someone.

Providers are free-tier only: Groq (`llama-3.3-70b-versatile`) or Gemini
(`gemini-2.0-flash`), called over plain `httpx` with no vendor SDK. With no key, a
deterministic rule-based advisor takes over — and it names no product at all, because
without a model to read the evidence we will not guess one.

In [ ]:
from backend.app.llm.advisor import build_user_prompt

print("=" * 78)
print("PROMPT AS THE MODEL RECEIVES IT")
print("=" * 78)
print(build_user_prompt(diagnosis, report, environment, evidence))

In [ ]:
from backend.app.llm.advisor import generate_plan

plan = generate_plan(diagnosis, report, environment, evidence, pipeline.provider, settings)

print(f"generated by : {plan.generated_by}")
print(f"urgency      : {plan.urgency.value.upper()}\n")
print(f"SUMMARY\n  {plan.summary}\n")

print("IMMEDIATE ACTIONS")
for step in plan.immediate_actions:
    citations = f"  {step.citations}" if step.citations else ""
    print(f"  {step.order}. [{step.timing}] {step.action}{citations}")

if plan.chemical_options:
    print("\nCHEMICAL OPTIONS (each carries a citation, or it was removed)")
    for option in plan.chemical_options:
        print(f"  - {option}")
else:
    print("\nCHEMICAL OPTIONS: none given")

print("\nCULTURAL PRACTICES")
for practice in plan.cultural_practices:
    print(f"  - {practice}")

print(f"\nMONITORING\n  {plan.monitoring}")
print("\nCAUTIONS")
for caution in plan.cautions:
    print(f"  ! {caution}")
if plan.unsupported_claims_removed:
    print(f"\n{plan.unsupported_claims_removed} uncited chemical claim(s) were stripped.")

## 5. One call, and the escalation ladder

`pipeline.run()` is what `POST /api/v1/diagnose` invokes. Running it across all five
samples shows the point of the whole design: **the same disease at different measured
severities produces a different mandated posture.**

In [ ]:
rows = []
for path in SAMPLES:
    response = pipeline.run(read_image(str(path)), environment)
    rows.append({
        "sample": path.stem.replace("leaf_", ""),
        "severity %": f"{response.severity.severity_pct:5.1f}",
        "necrotic %": f"{response.severity.necrotic_pct:5.1f}",
        "lesions": response.severity.lesion_count,
        "tier": response.severity.action_tier.value,
        "plan urgency": response.plan.urgency.value,
        "chemicals": len(response.plan.chemical_options),
        "ms": int(sum(response.elapsed_ms.values())),
    })

header = list(rows[0])
widths = {key: max(len(key), *(len(str(row[key])) for row in rows)) for key in header}
print("  ".join(key.ljust(widths[key]) for key in header))
print("  ".join("-" * widths[key] for key in header))
for row in rows:
    print("  ".join(str(row[key]).ljust(widths[key]) for key in header))

In [ ]:
response = pipeline.run(read_image(str(IMAGE_PATH)), environment, include_overlay=True)

print("STAGE LATENCY (ms)")
for stage, milliseconds in response.elapsed_ms.items():
    bar = "#" * max(int(milliseconds / 2), 1)
    print(f"  {stage:<12} {milliseconds:7.1f}  {bar}")
print(f"  {'TOTAL':<12} {sum(response.elapsed_ms.values()):7.1f}")

payload = response.model_dump_json()
print(f"\nThis is the exact JSON the PWA receives from POST /api/v1/diagnose "
      f"({len(payload)} bytes).")

## Where to go next

| Goal | Do this |
|------|---------|
| Real disease labels instead of placeholders | `scripts/download_plantvillage.py`, then `02_train_resnet50.ipynb` |
| Tune the severity thresholds against annotated leaves | `01_severity_lab.ipynb` |
| Move off the file store onto pgvector | `docker compose up -d postgres`, then `03_rag_pgvector.ipynb` |
| Turn on the LLM advisor (free) | Put a `GROQ_API_KEY` or `GEMINI_API_KEY` in `.env` and re-run — the provider is auto-detected |
| Serve it | `python -m uvicorn app.main:app --reload --app-dir backend`, then open `http://localhost:8000/docs` |

See `docs/PLAN.md` for the phase plan and the risk register.